# 00 - Environment Setup

## Objective

This notebook validates the Databricks environment and confirms that the project configuration, Spark session, Volume, raw data directory, and expected CSV files are available before running the data pipeline.

#### Load project configuration

In [0]:
# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")

#### Validate Spark and storage access

In [0]:
# Validate the Spark session

if spark is None:
    raise RuntimeError("Spark session is not available.")

print("Spark session is available.")


# Validate access to the project Volume

try:
    dbutils.fs.ls(cfg.VOLUME_PATH)
except Exception as exc:
    raise RuntimeError(
        f"Unable to access the configured Volume: {cfg.VOLUME_PATH}"
    ) from exc

print(f"Volume is accessible: {cfg.VOLUME_PATH}")

#### Validate project data directories

In [0]:
# Validate access to the raw data directory

try:
    raw_directory_contents = dbutils.fs.ls(cfg.RAW_PATH)
except Exception as exc:
    raise RuntimeError(
        f"Unable to access the raw data directory: {cfg.RAW_PATH}"
    ) from exc

print(f"Raw data directory is accessible: {cfg.RAW_PATH}")


# Validate access to the reference data directory

try:
    reference_directory_contents = dbutils.fs.ls(
        cfg.REFERENCE_PATH
    )
except Exception as exc:
    raise RuntimeError(
        f"Unable to access the reference data directory: "
        f"{cfg.REFERENCE_PATH}"
    ) from exc

print(
    f"Reference data directory is accessible: "
    f"{cfg.REFERENCE_PATH}"
)

# Validate access to the processed data directory

try:
    processed_directory_contents = dbutils.fs.ls(
        cfg.PROCESSED_PATH
    )
except Exception as exc:
    raise RuntimeError(
        f"Unable to access the processed data directory: "
        f"{cfg.PROCESSED_PATH}"
    ) from exc

print(
    f"Processed data directory is accessible: "
    f"{cfg.PROCESSED_PATH}"
)


#### Identify raw CSV files

In [0]:
# Identify the monthly flight CSV files

raw_csv_files = [
    file_info
    for file_info in raw_directory_contents
    if (
        not file_info.isDir()
        and file_info.name.lower().endswith(
            cfg.EXPECTED_FILE_EXTENSION.lower()
        )
    )
]

print(f"Raw CSV files found: {len(raw_csv_files)}")

#### Validate raw CSV files

In [0]:
# Validate the expected number of monthly flight files

if len(raw_csv_files) != cfg.EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Expected {cfg.EXPECTED_FILE_COUNT} raw CSV files, "
        f"but found {len(raw_csv_files)} in {cfg.RAW_PATH}."
    )

print(
    f"Expected number of raw files confirmed: "
    f"{cfg.EXPECTED_FILE_COUNT}"
)

#### Identify reference CSV files

In [0]:
# Identify the available reference CSV files

reference_csv_files = [
    file_info
    for file_info in reference_directory_contents
    if (
        not file_info.isDir()
        and file_info.name.lower().endswith(
            cfg.EXPECTED_FILE_EXTENSION.lower()
        )
    )
]

print(
    f"Reference CSV files found: "
    f"{len(reference_csv_files)}"
)

#### Validate reference CSV files

In [0]:
# Validate the expected number of reference files

if (
    len(reference_csv_files)
    != cfg.EXPECTED_REFERENCE_FILE_COUNT
):
    raise ValueError(
        f"Expected {cfg.EXPECTED_REFERENCE_FILE_COUNT} "
        f"reference CSV files, but found "
        f"{len(reference_csv_files)} in "
        f"{cfg.REFERENCE_PATH}."
    )

print(
    f"Expected number of reference files confirmed: "
    f"{cfg.EXPECTED_REFERENCE_FILE_COUNT}"
)

In [0]:
# Validate that all required reference files are available

available_reference_files = {
    file_info.name
    for file_info in reference_csv_files
}

missing_reference_files = sorted(
    set(cfg.EXPECTED_REFERENCE_FILES)
    - available_reference_files
)

unexpected_reference_files = sorted(
    available_reference_files
    - set(cfg.EXPECTED_REFERENCE_FILES)
)

if missing_reference_files:
    raise ValueError(
        "The following required reference files are missing: "
        + ", ".join(missing_reference_files)
    )

if unexpected_reference_files:
    raise ValueError(
        "The following unexpected reference files were found: "
        + ", ".join(unexpected_reference_files)
    )

print("All required reference files are available.")

#### Display available project files

In [0]:
# Display the raw files available for ingestion

print("Raw flight files:")

for file_info in sorted(
    raw_csv_files,
    key=lambda file: file.name
):
    print(f"- {file_info.name}")

In [0]:
# Display the reference files available for ingestion

print("Reference files:")

for file_info in sorted(
    reference_csv_files,
    key=lambda file: file.name
):
    print(f"- {file_info.name}")

#### Complete environment validation

In [0]:
print("Environment validation completed successfully.")